# Online Retail II — Data Cleaning (Day 2)

## Goal

Take the raw dataset from Day 1 and produce a clean, analysis-ready version saved to disk. Every cleaning decision is documented with rationale.

## Cleaning plan

Based on Day 1 exploration, we need to:
1. Load the raw data
2. Remove 'A'-prefix accounting adjustments
3. Handle cancelled orders ('C' prefix) — separate out for analysis
4. Deal with non-product StockCodes (POST, DOT, M, BANK CHARGES, etc.)
5. Handle nulls in Description
6. Handle missing Customer IDs (keep rows but flag them)
7. Remove zero or negative Prices that aren't cancellations
8. Remove exact duplicates
9. Engineer useful columns (Revenue, YearMonth, etc.)
10. Save the cleaned dataset

At each step we count how many rows we drop so the data loss is transparent.

In [9]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

# Load raw data
path = "/Users/edi/Documents/online-retail-analysis/data/online_retail_II.xlsx"
df1 = pd.read_excel(path, sheet_name="Year 2009-2010")
df2 = pd.read_excel(path, sheet_name="Year 2010-2011")
raw = pd.concat([df1, df2], ignore_index=True)

print(f"Raw dataset: {len(raw):,} rows")

Raw dataset: 1,067,371 rows


In [10]:
# Track cleaning losses step by step
cleaning_log = [{"step": "Raw data", "rows": len(raw), "dropped": 0}]

df = raw.copy()

# Ensure Invoice is string for consistent handling
df["Invoice"] = df["Invoice"].astype(str)
df["StockCode"] = df["StockCode"].astype(str)

print(f"Working copy: {len(df):,} rows")

Working copy: 1,067,371 rows


## Step 1: Remove accounting adjustments ('A' prefix)
These are bad-debt write-offs, not customer transactions. Keeping them would distort any revenue or customer metric.

In [11]:
before = len(df)
df = df[~df["Invoice"].str.startswith("A")].copy()
dropped = before-len(df)

cleaning_log.append({"step": "Remove 'A' adjustments", "rows": len(df), "dropped": dropped})
print(f"Dropped {dropped} accounting adjustment rows. Remaining: {len(df):,}")



Dropped 6 accounting adjustment rows. Remaining: 1,067,365


## Step 2: Separate cancellations

Cancellations (Invoice prefix 'C') are real business events but need to be analyzed separately — not stripped from the dataset. We split them into their own DataFrame for later analysis (return patterns, products most returned), and keep only completed sales in the main cleaning pipeline.

In [14]:
cancellations = df[df["Invoice"].str.startswith("C")].copy()
completed_sales = df[~df["Invoice"].str.startswith("C")].copy()

print(f"Cancellations: {len(cancellations):,} rows")
print(f"Completed sales: {len(completed_sales):,} rows")
print(f"\nCancellation rate: {len(cancellations)/(len(cancellations)+len(completed_sales))*100:.2f}%")

# Keep working on the completed_sales branch; handle cancellations at the end
df = completed_sales
cleaning_log.append({"step": "Split cancellations", "rows": len(df), "dropped": len(cancellations)})

Cancellations: 19,494 rows
Completed sales: 1,047,871 rows

Cancellation rate: 1.83%


## Step 3: Remove non-product StockCodes

Some StockCodes are not real products. For instance, they represent things like postage, gift cards, manual adjustments, or bank charges. They'll distort product-level analysis. Identify and remove them.

In [29]:
# Known non-product codes — identified from Day 1 exploration
NON_PRODUCT_CODES = {
    "POST",           # postage
    "DOT",            # dotcom postage
    "M",              # manual adjustments
    "C2",             # carriage
    "BANK CHARGES",   # bank fees
    "ADJUST",         # manual adjustments
    "AMAZONFEE",      # amazon fees
    "D",              # discounts
    "S",              # samples
    "PADS",           # pads to match cushions (not a sellable product)
    "CRUK",           # charity donation (CRUK = Cancer Research UK)
    "TEST001",        # test product
    "TEST002",        # test product
}

# Flag real products: any code NOT in the exclusion list
df["code_is_product"] = ~df["StockCode"].isin(NON_PRODUCT_CODES)

n_non_product = (~df["code_is_product"]).sum()
print(f"Rows flagged as non-product: {n_non_product:,}")
print(f"Rows flagged as product: {df['code_is_product'].sum():,}")

print("\nNon-product breakdown:")
print(df[~df["code_is_product"]]["StockCode"].value_counts())

Rows flagged as non-product: 4,608
Rows flagged as product: 1,043,263

Non-product breakdown:
StockCode
POST            1893
DOT             1443
M                883
C2               275
ADJUST            36
BANK CHARGES      35
PADS              18
TEST001           11
D                  5
AMAZONFEE          4
S                  3
TEST002            2
Name: count, dtype: int64


In [30]:
# Drop non-product codes (postage, bank charges, manual entries, etc.)
before = len(df)
df = df[df["code_is_product"]].copy()
df = df.drop(columns=["code_is_product"])
dropped = before - len(df)

cleaning_log.append({"step": "Drop non-product codes", "rows": len(df), "dropped": dropped})
print(f"Dropped {dropped:,} non-product rows. Remaining: {len(df):,}")

Dropped 4,608 non-product rows. Remaining: 1,043,263


## Step 4: Handle missing Descriptions

Some rows have null Description. These tend to be test data, zero-price rows, or broken entries. Since Description gives us the product name, losing it means we can't meaningfully use the row for product analysis.

In [32]:
before = len(df)
n_null_desc = df["Description"].isnull().sum()
print(f"Rows with null Description: {n_null_desc:,}")

# Inspect a few before deciding
display(df[df["Description"].isnull()].head())

Rows with null Description: 4,369


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.00,NaN,United Kingdom
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.00,NaN,United Kingdom
3161,489659,21350,NaN,230,2009-12-01 17:39:00,0.00,NaN,United Kingdom
3731,489781,84292,NaN,17,2009-12-02 11:45:00,0.00,NaN,United Kingdom
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.00,NaN,United Kingdom


### Inspecting null Description rows

4,369 rows have null `Description`. Inspection reveals a consistent pattern:
- `Price` is always 0.00
- `Customer ID` is always null  
- `Quantity` values are extreme and often negative (e.g., -770, +230)

These appear to be **inventory adjustments** — internal stock corrections made by warehouse staff, not customer purchases. Dropping them from the sales analysis is safe; they'd otherwise distort revenue (all zeros) and quantity-based metrics.

In [33]:
# Drop rows with missing Description
df = df.dropna(subset=["Description"]).copy()
dropped = before - len(df)

# Also strip whitespace from Description 
df["Description"] = df["Description"].str.strip()

cleaning_log.append({"step": "Drop null Description", "rows": len(df), "dropped": dropped})
print(f"Dropped {dropped:,} null-description rows. Remaining: {len(df):,}")

Dropped 4,369 null-description rows. Remaining: 1,038,894


## Step 5: Remove zero or negative Price rows

After removing non-product codes and cancellations, any remaining zero or negative Price is suspicious — probably a data entry error or a non-sale test row.

In [35]:
before = len(df)
weird_prices = df[df["Price"] <= 0]

print(f"Rows with Price <= 0: {len(weird_prices):,}")
print(f"\nSample:")
display(weird_prices.head())

Rows with Price <= 0: 1,805

Sample:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.00,NaN,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,0.00,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.00,NaN,United Kingdom
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.00,NaN,United Kingdom
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,0.00,NaN,United Kingdom


In [36]:
df = df[df["Price"] > 0].copy()
dropped = before - len(df)

cleaning_log.append({"step": "Drop Price <= 0", "rows": len(df), "dropped": dropped})
print(f"Dropped {dropped:,} zero/negative-price rows. Remaining: {len(df):,}")

Dropped 1,805 zero/negative-price rows. Remaining: 1,037,089


## Step 6: Remove zero or negative Quantity

After removing cancellations (C-prefix), any remaining negative/zero quantity is likely a data entry error.

In [37]:
before = len(df)
df = df[df["Quantity"] > 0].copy()
dropped = before - len(df)

cleaning_log.append({"step": "Drop Quantity <= 0", "rows": len(df), "dropped": dropped})
print(f"Dropped {dropped:,} zero/negative-quantity rows. Remaining: {len(df):,}")

Dropped 0 zero/negative-quantity rows. Remaining: 1,037,089


## Step 7: Remove exact duplicate rows

Same invoice + same product + same quantity + same price + same time = almost certainly a duplicate insertion, not a real second purchase.

In [38]:
before = len(df)
df = df.drop_duplicates().copy()
dropped = before - len(df)

cleaning_log.append({"step": "Drop exact duplicates", "rows": len(df), "dropped": dropped})
print(f"Dropped {dropped:,} exact duplicates. Remaining: {len(df):,}")

Dropped 33,664 exact duplicates. Remaining: 1,003,425


## Step 8: Handle missing Customer ID

About 20-25% of rows have null Customer ID. These rows are still valid sales (real products bought), but we can't tie them to a specific customer for segmentation analysis.

**Decision:** keep them but flag them. This way:
- Revenue/product/time/country analysis uses *all* rows
- Customer-level analysis filters to rows where Customer ID is known

In [39]:
n_missing = df["Customer ID"].isnull().sum()
pct_missing = n_missing / len(df) * 100
print(f"Rows with null Customer ID: {n_missing:,} ({pct_missing:.1f}%)")

df["customer_known"] = df["Customer ID"].notnull()

# For the known customer rows, convert ID to integer (nicer than float)
df.loc[df["customer_known"], "Customer ID"] = df.loc[df["customer_known"], "Customer ID"].astype(int)

Rows with null Customer ID: 226,843 (22.6%)


## Step 9: Feature engineering

Add derived columns that downstream analyses will need:
- **Revenue** = Quantity × Price (the single most important business metric)
- **InvoiceDate** split into Year, Month, YearMonth, DayOfWeek, Hour for time-series viz
- Rename columns to snake_case so they're SQL-friendly (no spaces, lowercase)

In [45]:
df["Revenue"] = df["Quantity"] * df["Price"]

# Date parts
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["Year"] = df["InvoiceDate"].dt.year
df["Month"] = df["InvoiceDate"].dt.month
df["YearMonth"] = df["InvoiceDate"].dt.to_period("M").astype(str)
df["DayOfWeek"] = df["InvoiceDate"].dt.day_name()
df["Hour"] = df["InvoiceDate"].dt.hour

# Rename columns to snake_case
df = df.rename(columns={
    "Invoice": "invoice",
    "StockCode": "stock_code",
    "Description": "description",
    "Quantity": "quantity",
    "InvoiceDate": "invoice_date",
    "Price": "price",
    "Customer ID": "customer_id",
    "Country": "country",
    "Revenue": "revenue",
    "Year": "year",
    "Month": "month",
    "YearMonth": "year_month",
    "DayOfWeek": "day_of_week",
    "Hour": "hour"
})

print(f"Final clean dataset: {len(df):,} rows, {df.shape[1]} columns")
df.head()

Final clean dataset: 1,003,425 rows, 15 columns


,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country,customer_known,revenue,year,month,year_month,day_of_week,hour
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,"13,085.00",United Kingdom,True,83.40,2009,12,2009-12,Tuesday,7
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom,True,81.00,2009,12,2009-12,Tuesday,7
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom,True,81.00,2009,12,2009-12,Tuesday,7
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,"13,085.00",United Kingdom,True,100.80,2009,12,2009-12,Tuesday,7
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,"13,085.00",United Kingdom,True,30.00,2009,12,2009-12,Tuesday,7


## Step 10: Quality checks before saving

Quick sanity checks on the cleaned data.

In [46]:
print("Revenue summary:")
print(df["revenue"].describe())

print(f"\nDate range: {df['invoice_date'].min()} to {df['invoice_date'].max()}")
print(f"Unique customers: {df[df['customer_known']]['customer_id'].nunique():,}")
print(f"Unique products: {df['stock_code'].nunique():,}")
print(f"Unique invoices: {df['invoice'].nunique():,}")
print(f"Countries: {df['country'].nunique()}")

# Check for any remaining nulls
print(f"\nRemaining nulls per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Revenue summary:
count   1,003,425.00
mean           19.58
std           199.98
min             0.06
25%             4.13
50%            10.08
75%            17.70
max       168,469.60
Name: revenue, dtype: float64

Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00
Unique customers: 5,852
Unique products: 4,904
Unique invoices: 39,519
Countries: 43

Remaining nulls per column:
customer_id    226843
dtype: int64


## Cleaning log

Transparent record of every row we dropped and why.

In [47]:
log_df = pd.DataFrame(cleaning_log)
log_df["pct_of_raw"] = (log_df["rows"] / log_df.iloc[0]["rows"] * 100).round(1)
log_df["cumulative_dropped"] = log_df["dropped"].cumsum()
log_df

,step,rows,dropped,pct_of_raw,cumulative_dropped
0,Raw data,1067371,0,100.00,0
1,Remove 'A' adjustments,1067365,6,100.00,6
2,Split cancellations,1047871,19494,98.20,19500
3,Drop non-product codes,1043263,4608,97.70,24108
4,Drop null Description,1038894,4369,97.30,28477
5,Drop Price <= 0,1037089,1805,97.20,30282
6,Drop Quantity <= 0,1037089,0,97.20,30282
7,Drop exact duplicates,1003425,33664,94.00,63946


## Save cleaned datasets

Two files:
- `sales_clean.csv` — the cleaned completed sales (main analysis dataset)
- `cancellations.csv` — separated cancellations for return analysis

In [48]:
from pathlib import Path

out_dir = Path("/Users/edi/Documents/data")
out_dir.mkdir(exist_ok=True)

df.to_csv(out_dir / "sales_clean.csv", index=False)
cancellations.to_csv(out_dir / "cancellations.csv", index=False)

print(f"Saved sales_clean.csv: {len(df):,} rows")
print(f"Saved cancellations.csv: {len(cancellations):,} rows")

Saved sales_clean.csv: 1,003,425 rows
Saved cancellations.csv: 19,494 rows


## Summary

Starting with ~1.07M raw rows, our cleaning pipeline produced a final analysis-ready dataset by:
- Removing 6 accounting adjustment entries
- Separating ~22K cancellations for separate analysis
- Dropping ~10K non-product SKU rows (postage, bank charges, etc.)
- Dropping rows with null descriptions, zero/negative prices, and zero quantities
- Removing exact duplicates
- Flagging (not dropping) the ~20-25% of rows with missing Customer IDs
- Engineering useful date and revenue columns

The final dataset is ready for the business-question analysis in Notebook 03.